In [ ]:
from manim import *
import numpy as np

class SVDDeepDive(Scene):
    """
    A Manim scene that visually explains the Singular Value Decomposition (SVD)
    through its geometric properties.
    """
    def construct(self):
        # 1. Setup Configuration & Layout
        self.setup_constants()
        self.setup_layout()
        
        # 2. Run Animation Sequence
        self.anim_decomposition()   # Property 1: Decomposition steps
        self.anim_determinant()     # Property 2: Determinant as Area
        self.anim_rank()            # Property 3: Rank Collapse
        self.anim_approximation()   # Property 4: Low-Rank Approximation
        
        # 3. Conclusion
        self.outro()

    def setup_constants(self):
        """Define colors, matrix values, and mathematical constants."""
        # Color Palette
        self.COL_SIGMA1 = YELLOW
        self.COL_SIGMA2 = BLUE
        self.COL_V = RED       
        self.COL_U = GREEN     
        self.COL_GHOST = GREY  
        self.COL_TEXT = WHITE
        
        # Singular Values
        self.sigma1_val = 2.5 
        self.sigma2_val = 1.0 
        
        # Rotation Angles
        self.theta_v_deg = -45
        self.theta_v = self.theta_v_deg * DEGREES
        self.theta_u_deg = 30
        self.theta_u = self.theta_u_deg * DEGREES
        
        # Matrices Definition
        # V^T (Input Rotation)
        self.V_T = np.array([
            [np.cos(self.theta_v), -np.sin(self.theta_v)],
            [np.sin(self.theta_v), np.cos(self.theta_v)]
        ])
        
        # U (Output Rotation)
        self.U = np.array([
            [np.cos(self.theta_u), -np.sin(self.theta_u)],
            [np.sin(self.theta_u), np.cos(self.theta_u)]
        ])
        
        # Sigma (Scaling)
        self.S = np.array([[self.sigma1_val, 0], [0, self.sigma2_val]])
        
        # Full Matrix A
        self.A = self.U @ self.S @ self.V_T
        
        # State tracking for text management
        self.current_text_group = VGroup()
        
        # LABEL POSITION: Top-Left of the Geometry Panel
        # Separator is at x = -1. Top is y = 4.
        # "Left 1.5 units" (relative to separator) -> x = -1 + 1.5 = 0.5
        # "Bottom 1.5 units" (relative to top) -> y = 4 - 1.5 = 2.5
        self.LABEL_POS = np.array([0.5, 2.5, 0])

    def setup_layout(self):
        """Initialize the split-screen layout and static geometric elements."""
        # Visual Separator (Left: Text, Right: Geometry)
        self.separator = Line(UP*4, DOWN*4).shift(LEFT*1)
        self.play(Create(self.separator))
        
        # Main Title
        self.title = Title("Geometric Interpretation of SVD").scale(0.8)
        self.add(self.title)

        # 1. Ghost Plane (Static Reference Frame)
        self.ghost_plane = NumberPlane(
            x_range=[-6, 6, 1], y_range=[-6, 6, 1],
            background_line_style={"stroke_color": self.COL_GHOST, "stroke_opacity": 0.2},
            axis_config={"stroke_color": self.COL_GHOST, "stroke_opacity": 0.3}
        ).scale(0.6).shift(RIGHT * 3.5 + DOWN * 0.5)
        
        # 2. Active Plane (Will be transformed)
        self.active_plane = NumberPlane(
            x_range=[-6, 6, 1], y_range=[-6, 6, 1],
            background_line_style={"stroke_opacity": 0.4}
        ).scale(0.6).shift(RIGHT * 3.5 + DOWN * 0.5)

        # 3. Origin Point (O)
        self.origin_point = Dot(self.active_plane.get_center(), color=WHITE, radius=0.08)
        self.origin_lbl = MathTex("O", font_size=24).next_to(self.origin_point, DL, buff=0.1)

        # 3. Unit Circles (Ghost & Active)
        self.ghost_circle = Circle(radius=0.6, color=self.COL_GHOST, stroke_opacity=0.3, stroke_width=2).move_to(self.active_plane.get_center())
        self.unit_circle = Circle(radius=0.6, color=WHITE, stroke_opacity=0.8).move_to(self.active_plane.get_center())

        # 4. Basis Vectors
        self.vec_i = Vector([1, 0], color=self.COL_V).shift(self.active_plane.get_center()).scale(0.6)
        self.vec_j = Vector([0, 1], color=self.COL_V).shift(self.active_plane.get_center()).scale(0.6)
        
        # Labels
        self.label_i = MathTex(r"v_1", color=self.COL_V).next_to(self.vec_i, UP, buff=0.05).scale(0.6)
        self.label_j = MathTex(r"v_2", color=self.COL_V).next_to(self.vec_j, LEFT, buff=0.05).scale(0.6)

        # Animate Setup
        self.play(
            FadeIn(self.ghost_plane), 
            Create(self.ghost_circle),
            Create(self.active_plane),
            FadeIn(self.origin_point), Write(self.origin_lbl),
            Create(self.unit_circle),
            GrowArrow(self.vec_i), GrowArrow(self.vec_j), 
            Write(self.label_i), Write(self.label_j)
        )

    def update_text_area(self, mobjects_list):
        """
        Manages text on the left panel. 
        Strictly enforces width limits to prevent overflow.
        """
        # Clear previous
        if self.current_text_group:
             self.play(FadeOut(self.current_text_group), run_time=0.5)
        
        # Create group, align left
        new_group = VGroup(*mobjects_list).arrange(DOWN, aligned_edge=LEFT, buff=0.25)
        
        # --- Overflow Handling ---
        max_width = 5.0
        if new_group.width > max_width:
            new_group.scale_to_fit_width(max_width)
        
        # Position logic
        new_group.to_edge(LEFT, buff=0.5)
        
        # Prevent overlap with title
        if new_group.get_top()[1] > 2.8: 
            new_group.to_edge(UP, buff=1.5)
            new_group.to_edge(LEFT, buff=0.5)
            
        self.play(FadeIn(new_group), run_time=0.8)
        self.current_text_group = new_group

    def anim_decomposition(self):
        """Property 1: The Step-by-Step Decomposition"""
        
        # --- Part 1: Input Rotation (V^T) ---
        t1 = Tex(r"\textbf{1. Input Rotation } ($V^T$)", font_size=36)
        t1_desc = VGroup(
            Tex(r"First, we align the grid.", font_size=28),
            Tex(r"The matrix $V^T$ rotates the basis", font_size=28),
            Tex(r"to align with the principal axes.", font_size=28),
            Tex(r"Note the angle relative to the", font_size=28),
            Tex(r"static reference line.", font_size=28)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        t1_math = MathTex(r"x' = V^T x", color=self.COL_V, font_size=32)
        
        self.update_text_area([t1, t1_desc, t1_math])

        # Matrix Label - Top Left of Geometry Panel
        self.matrix_label = MathTex(r"V^T", font_size=60).move_to(self.LABEL_POS).set_color(self.COL_V)
        self.play(Write(self.matrix_label))

        # 1. Create STATIC Reference Line (Original X-axis)
        center = self.active_plane.get_center()
        ref_line_v = DashedLine(center, center + RIGHT*3, color=GRAY)
        self.add(ref_line_v)

        # Animation: Rotate
        self.play(
            self.active_plane.animate.apply_matrix(self.V_T),
            Rotate(self.unit_circle, self.theta_v, about_point=center),
            Rotate(self.vec_i, self.theta_v, about_point=center),
            Rotate(self.vec_j, self.theta_v, about_point=center),
            Rotate(self.label_i, self.theta_v, about_point=center),
            Rotate(self.label_j, self.theta_v, about_point=center),
            run_time=2
        )

        # Angle Measurement: Between Static Ref Line and New Vector
        angle_arc = Arc(
            radius=0.8, 
            start_angle=0, 
            angle=self.theta_v, 
            arc_center=center, 
            color=self.COL_V
        )
        angle_val = MathTex(f"{self.theta_v_deg}^\circ", color=self.COL_V, font_size=24).next_to(angle_arc, RIGHT, buff=0.1).set_background_stroke(color=BLACK, width=5)
        
        self.play(Create(angle_arc), Write(angle_val))
        self.wait(1)
        self.play(FadeOut(angle_arc), FadeOut(angle_val), FadeOut(ref_line_v))


        # --- Part 2: Scaling (Sigma) - DEEP DIVE ---
        t2 = Tex(r"\textbf{2. Scaling } ($\Sigma$)", font_size=36)
        t2_desc = VGroup(
            Tex(r"This is the core of SVD.", font_size=28),
            Tex(r"We stretch along the new axes.", font_size=28),
            Tex(r"Compare \textbf{New Length} ($\sigma$)", font_size=28),
            Tex(r"to the \textbf{Original Length} (1.0)", font_size=28),
            Tex(r"starting from Origin $O$.", font_size=28)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        t2_math_1 = MathTex(r"\sigma_1 = " + str(self.sigma1_val) + r" \text{ (Major)}", font_size=32).set_color(self.COL_SIGMA1)
        t2_math_2 = MathTex(r"\sigma_2 = " + str(self.sigma2_val) + r" \text{ (Minor)}", font_size=32).set_color(self.COL_SIGMA2)
        
        self.update_text_area([t2, t2_desc, t2_math_1, t2_math_2])

        self.play(Transform(self.matrix_label, MathTex(r"\Sigma", font_size=60).move_to(self.LABEL_POS).set_color(self.COL_SIGMA1)))

        # Ghost Vectors: Visualize the UNIT length along the current axes
        scale = 0.6
        # Directions of the current axes (after V^T rotation)
        v1_dir = np.array([np.cos(self.theta_v), np.sin(self.theta_v), 0])
        v2_dir = np.array([np.cos(self.theta_v + PI/2), np.sin(self.theta_v + PI/2), 0])

        ghost_vec_1 = DashedLine(center, center + v1_dir * scale, color=GRAY, stroke_opacity=0.8)
        
        self.play(Create(ghost_vec_1))

        # Animation: Apply Scaling
        self.play(
            self.active_plane.animate.apply_matrix(self.S),
            self.unit_circle.animate.scale([self.sigma1_val, self.sigma2_val, 1], about_point=center).set_color(YELLOW),
            self.vec_i.animate.scale([self.sigma1_val, 1, 1], about_point=center).set_color(self.COL_SIGMA1), 
            self.vec_j.animate.scale([self.sigma2_val, 1, 1], about_point=center).set_color(self.COL_SIGMA2),
            # Labels follow tips
            self.label_i.animate.move_to(self.active_plane.c2p(self.sigma1_val*np.cos(self.theta_v), self.sigma1_val*np.sin(self.theta_v))).shift(RIGHT*0.2),
            self.label_j.animate.move_to(self.active_plane.c2p(self.sigma2_val*np.cos(self.theta_v+PI/2), self.sigma2_val*np.sin(self.theta_v+PI/2))).shift(UP*0.2),
            run_time=2
        )
        
        # Geometric Insight: Measure Scaling Factor
        
        # 1. Brace for Unit Length (Reference)
        ortho_v1 = np.array([v1_dir[1], -v1_dir[0], 0])
        brace_unit = Brace(ghost_vec_1, direction=ortho_v1, buff=0.05)
        txt_unit = brace_unit.get_text("1.0").scale(0.6).set_color(GRAY).set_background_stroke(color=BLACK, width=5)
        
        # 2. Brace for Sigma 1 Length (Yellow)
        v1_end = self.vec_i.get_end()
        line_sigma1 = Line(center, v1_end)
        brace_sigma1 = Brace(line_sigma1, direction=ortho_v1, buff=0.45) 
        brace_sigma1.set_color(self.COL_SIGMA1)
        txt_sigma1 = brace_sigma1.get_text(f"$\\sigma_1 = {self.sigma1_val}$").scale(0.8).set_color(self.COL_SIGMA1).set_background_stroke(color=BLACK, width=5)
        
        # 3. Brace for Sigma 2 Length (Blue)
        v2_end = self.vec_j.get_end()
        line_sigma2 = Line(center, v2_end)
        ortho_v2 = np.array([v2_dir[1], -v2_dir[0], 0]) 
        brace_sigma2 = Brace(line_sigma2, direction=ortho_v2, buff=0.1)
        brace_sigma2.set_color(self.COL_SIGMA2)
        txt_sigma2 = brace_sigma2.get_text(f"$\\sigma_2 = {self.sigma2_val}$").scale(0.6).set_color(self.COL_SIGMA2).set_background_stroke(color=BLACK, width=5)

        self.play(
            Create(brace_unit), Write(txt_unit),
            Create(brace_sigma1), Write(txt_sigma1),
            Create(brace_sigma2), Write(txt_sigma2)
        )
        self.wait(2)
        
        # Cleanup Scaling visuals
        self.play(
            FadeOut(brace_unit), FadeOut(txt_unit),
            FadeOut(brace_sigma1), FadeOut(txt_sigma1),
            FadeOut(brace_sigma2), FadeOut(txt_sigma2),
            FadeOut(ghost_vec_1)
        )

        # --- Part 3: Output Rotation (U) ---
        t3 = Tex(r"\textbf{3. Output Rotation } ($U$)", font_size=36)
        t3_desc = VGroup(
            Tex(r"Finally, rotate to the output frame.", font_size=28),
            Tex(r"We rotate the entire grid.", font_size=28),
            Tex(r"Compare the old axis direction", font_size=28),
            Tex(r"to the new one.", font_size=28)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        t3_math = MathTex(r"A = U \Sigma V^T", font_size=32, color=self.COL_U)
        
        self.update_text_area([t3, t3_desc, t3_math])

        self.play(Transform(self.matrix_label, MathTex(r"U", font_size=60).move_to(self.LABEL_POS).set_color(self.COL_U)))

        # 1. Create STATIC Reference Line (Current Major Axis Direction)
        # This line originates from Origin and points along the CURRENT v1 direction
        ref_line_u = DashedLine(center, center + v1_dir * 3, color=GRAY, stroke_opacity=0.5)
        self.add(ref_line_u)

        # Animation: Rotate everything EXCEPT the ref_line
        self.play(
            self.active_plane.animate.apply_matrix(self.U),
            Rotate(self.unit_circle, self.theta_u, about_point=center),
            Rotate(self.vec_i, self.theta_u, about_point=center),
            Rotate(self.vec_j, self.theta_u, about_point=center),
            Rotate(self.label_i, self.theta_u, about_point=center),
            Rotate(self.label_j, self.theta_u, about_point=center),
            run_time=2
        )
        
        # Angle Measurement: JOINED to reference line
        # Start angle is theta_v (direction of ref line)
        # Angle change is theta_u
        angle_arc_u = Arc(
            radius=1.0, 
            start_angle=self.theta_v, 
            angle=self.theta_u,       
            arc_center=center, 
            color=self.COL_U
        )
        angle_val_u = MathTex(f"{self.theta_u_deg}^\circ", color=self.COL_U, font_size=24).next_to(angle_arc_u, RIGHT, buff=0.1).set_background_stroke(color=BLACK, width=5)
        
        self.play(Create(angle_arc_u), Write(angle_val_u))
        self.wait(1)
        self.play(FadeOut(angle_arc_u), FadeOut(angle_val_u), FadeOut(ref_line_u))

        self.final_eq = MathTex(r"A = U \Sigma V^T", font_size=48).move_to(self.LABEL_POS)
        self.play(ReplacementTransform(self.matrix_label, self.final_eq))
        self.wait(2)

    def anim_determinant(self):
        """Property 2: Determinant as Area Scaling"""
        d1 = Tex(r"\textbf{Determinant = Area}", font_size=36)
        d_desc = VGroup(
            Tex(r"Determinant measures area change.", font_size=28),
            Tex(r"Only $\Sigma$ changes the area.", font_size=28),
            Tex(r"Area of the parallelogram spanned", font_size=28),
            Tex(r"by the vectors is $\sigma_1 \times \sigma_2$.", font_size=28)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        d_math = MathTex(r"|\det(A)| = \sigma_1 \times \sigma_2", font_size=32)
        d_math[0][6:8].set_color(self.COL_SIGMA1)
        d_math[0][9:11].set_color(self.COL_SIGMA2)
        
        self.update_text_area([d1, d_desc, d_math])

        # Geometric Proof
        center = self.active_plane.get_center()
        
        # CORRECT POLYGON CONSTRUCTION:
        p0 = center
        p1 = self.vec_i.get_end()
        p3 = self.vec_j.get_end()
        p2 = p1 + (p3 - center) 
        
        trans_rect = Polygon(p0, p1, p2, p3, color=WHITE, fill_color=TEAL, fill_opacity=0.5)
        
        # Ghost Square (1x1 unit)
        u_vec1 = (p1 - center) / self.sigma1_val
        u_vec2 = (p3 - center) / self.sigma2_val
        
        g0 = center
        g1 = center + u_vec1
        g3 = center + u_vec2
        g2 = g1 + (g3 - center)
        
        ghost_square = Polygon(g0, g1, g2, g3, color=self.COL_GHOST, fill_opacity=0.3, stroke_width=2)
        lbl_ghost = Tex("Area=1", font_size=24, color=GRAY).next_to(ghost_square, UP, buff=0.1).set_background_stroke(color=BLACK, width=5)

        self.play(FadeIn(ghost_square), Write(lbl_ghost))
        self.play(TransformFromCopy(ghost_square, trans_rect))
        
        area_math = MathTex(
            r"\text{Area} &= " + str(self.sigma1_val) + r" \times " + str(self.sigma2_val) + r"\\"
            r"&= " + str(self.sigma1_val * self.sigma2_val)
        ).scale(0.7).next_to(trans_rect, DOWN).set_background_stroke(color=BLACK, width=5)
        
        self.play(Write(area_math))
        self.wait(3)
        
        # Clean up
        self.play(
            FadeOut(ghost_square), FadeOut(trans_rect), FadeOut(area_math), FadeOut(lbl_ghost),
            FadeOut(self.unit_circle), FadeOut(self.ghost_circle), FadeOut(self.active_plane),
            FadeOut(self.vec_i), FadeOut(self.vec_j), FadeOut(self.label_i), FadeOut(self.label_j),
            FadeOut(self.final_eq), FadeOut(self.origin_point), FadeOut(self.origin_lbl)
        )

    def anim_rank(self):
        """Property 3: Rank and Dimensional Collapse"""
        r1 = Tex(r"\textbf{Rank \& The Collapse}", font_size=36)
        r_desc = VGroup(
            Tex(r"Rank = number of non-zero $\sigma$.", font_size=28),
            Tex(r"If $\sigma_2 \to 0$, the grid flattens.", font_size=28),
            Tex(r"The 2D area becomes a 1D line.", font_size=28)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        r_math = MathTex(r"\text{Rank } 2 \to 1", font_size=32, color=RED)
        
        self.update_text_area([r1, r_desc, r_math])

        # Rank Plane
        rank_plane = NumberPlane(x_range=[-5,5], y_range=[-5,5]).scale(0.6).shift(RIGHT*3.5 + DOWN*0.5)
        self.play(FadeIn(rank_plane))

        # Dynamic Grid
        grid_lines = VGroup()
        for x in np.linspace(-2, 2, 5):
            l = Line(np.array([x, -2, 0]), np.array([x, 2, 0]), color=BLUE_E, stroke_opacity=0.6)
            grid_lines.add(l)
        for y in np.linspace(-2, 2, 5):
            l = Line(np.array([-2, y, 0]), np.array([2, y, 0]), color=BLUE_E, stroke_opacity=0.6)
            grid_lines.add(l)
            
        self.add(grid_lines)

        def get_matrix(s2):
            return self.U @ np.array([[self.sigma1_val, 0], [0, s2]]) @ self.V_T

        def update_grid(mob):
            s2 = s2_tracker.get_value()
            mat = get_matrix(s2)
            idx = 0
            
            # Verticals
            for x in np.linspace(-2, 2, 5):
                start = np.array([x, -2]) 
                end = np.array([x, 2])
                new_start = rank_plane.c2p(*(mat @ start))
                new_end = rank_plane.c2p(*(mat @ end))
                mob[idx].put_start_and_end_on(new_start, new_end)
                idx += 1
            # Horizontals
            for y in np.linspace(-2, 2, 5):
                start = np.array([-2, y])
                end = np.array([2, y])
                new_start = rank_plane.c2p(*(mat @ start))
                new_end = rank_plane.c2p(*(mat @ end))
                mob[idx].put_start_and_end_on(new_start, new_end)
                idx += 1

        s2_tracker = ValueTracker(self.sigma2_val)
        s2_display = Variable(self.sigma2_val, MathTex(r"\sigma_2"), num_decimal_places=2).move_to(self.LABEL_POS).set_color(self.COL_SIGMA2)
        s2_display.tracker = s2_tracker
        self.add(s2_display)
        
        grid_lines.add_updater(update_grid)
        
        # Brace showing collapse
        squeeze_brace = Brace(Line(ORIGIN, UP), direction=RIGHT, color=self.COL_SIGMA2)
        def update_brace(mob):
            s2 = s2_tracker.get_value()
            mat = get_matrix(s2)
            center = rank_plane.c2p(0,0,0)
            edge_pt = mat @ np.array([0, 2]) 
            edge = rank_plane.c2p(*edge_pt)
            l = Line(center, edge)
            if np.linalg.norm(edge-center) > 0.1:
                 mob.become(Brace(l, direction=RIGHT, buff=0).set_color(self.COL_SIGMA2))
            else:
                 mob.become(Vector(ORIGIN).set_opacity(0)) 

        squeeze_brace.add_updater(update_brace)
        self.add(squeeze_brace)

        self.wait(0.5)
        self.play(s2_tracker.animate.set_value(0.0), run_time=5, rate_func=linear)
        
        line_col_space = DashedLine(
            start=rank_plane.c2p(*(self.U[:,0] * -4)),
            end=rank_plane.c2p(*(self.U[:,0] * 4)),
            color=YELLOW, stroke_width=4
        )
        t_col_space = Tex(r"Rank 1: Line", color=YELLOW, font_size=30).next_to(line_col_space, UP).set_background_stroke(color=BLACK, width=5)
        
        self.play(Create(line_col_space), Write(t_col_space))
        self.wait(2)
        
        grid_lines.remove_updater(update_grid)
        squeeze_brace.remove_updater(update_brace)
        self.play(FadeOut(grid_lines), FadeOut(squeeze_brace), FadeOut(line_col_space), FadeOut(t_col_space), FadeOut(rank_plane), FadeOut(s2_display))

    def anim_approximation(self):
        """Property 4: Low-Rank Approximation"""
        l1 = Tex(r"\textbf{4. Low-Rank Approximation}", font_size=36)
        l_desc = VGroup(
            Tex(r"Keep largest $\sigma$, drop small ones.", font_size=28),
            Tex(r"Geometrically, project data", font_size=28),
            Tex(r"onto the major axis.", font_size=28),
            Tex(r"The removed part is 'Error'.", font_size=28, color=BLUE),
            Tex(r"This removes 'noise' ($\sigma_2$).", font_size=28, color=BLUE)
        ).arrange(DOWN, aligned_edge=LEFT)
        
        l_math = MathTex(r"A \approx \sigma_1 u_1 v_1^T", font_size=32)
        
        self.update_text_area([l1, l_desc, l_math])

        approx_plane = NumberPlane(x_range=[-5,5], y_range=[-5,5], background_line_style={"stroke_opacity": 0.2}).scale(0.6).shift(RIGHT*3.5 + DOWN*0.5)
        self.play(FadeIn(approx_plane))
        
        # Exact Calculation - REDUCED VECTOR SIZE TO FIT SCREEN
        x_in = np.array([0.8, 0.5]) 
        b_exact = self.A @ x_in
        
        vec_exact = Vector(b_exact, color=YELLOW).shift(approx_plane.get_center()).scale(0.6)
        dot_exact = Dot(vec_exact.get_end(), color=YELLOW)
        lbl_exact = MathTex("Ax", color=YELLOW).next_to(dot_exact, UP, buff=0.1).set_background_stroke(color=BLACK, width=5)

        # Approx Calculation
        S_k = np.array([[self.sigma1_val, 0], [0, 0]])
        A_k = self.U @ S_k @ self.V_T
        b_approx = A_k @ x_in
        
        vec_approx = Vector(b_approx, color=RED).shift(approx_plane.get_center()).scale(0.6)
        dot_approx = Dot(vec_approx.get_end(), color=RED)
        lbl_approx = MathTex("A_1 x", color=RED).next_to(dot_approx, DOWN, buff=0.1).set_background_stroke(color=BLACK, width=5)
        
        u1_vec = self.U[:, 0]
        subspace_line = DashedLine(
            start=approx_plane.c2p(*(u1_vec * -4)), 
            end=approx_plane.c2p(*(u1_vec * 4)),
            color=WHITE, stroke_opacity=0.3
        )

        self.play(Create(subspace_line))
        self.play(GrowArrow(vec_exact), FadeIn(dot_exact), Write(lbl_exact))
        self.wait(1)
        
        self.play(GrowArrow(vec_approx), FadeIn(dot_approx), Write(lbl_approx))
        
        # Error Visualization
        err_line = DashedLine(start=vec_exact.get_end(), end=vec_approx.get_end(), color=BLUE)
        err_brace = Brace(err_line, direction=RIGHT, color=BLUE)
        err_lbl = err_brace.get_text("Error").scale(0.6).set_color(BLUE).set_background_stroke(color=BLACK, width=5)
        
        right_angle = RightAngle(
            Line(approx_plane.get_center(), vec_approx.get_end()),
            Line(vec_approx.get_end(), vec_exact.get_end()),
            length=0.2, quadrant=(-1, 1)
        )

        self.play(Create(err_line), Create(right_angle))
        self.play(Create(err_brace), Write(err_lbl))
        
        # FIXED: Position "final_insight" BELOW the brace to avoid overlap
        final_insight = Tex(r"This removes $\sigma_2$ (noise).", color=BLUE, font_size=24).next_to(err_brace, DOWN, buff=0.2).set_background_stroke(color=BLACK, width=5)
        self.play(Write(final_insight))
        self.wait(3)

    def outro(self):
        """Final Summary Slide"""
        # FIXED: Moved to bottom edge
        outro_text = Tex(
            r"SVD separates data into: ",
            r"Directions ($U, V^T$) ",
            r"\& Importances ($\Sigma$)",
            color=WHITE
        ).to_edge(DOWN, buff=1.0).scale(0.8)
        
        self.play(Write(outro_text))
        self.wait(3)


%manim -qk -v warning SVDDeepDive

Manim Community v0.19.0